# Module 1 Homework: Docker & SQL

In this homework we'll prepare the environment and practice
Docker and SQL

When submitting your homework, you will also need to include
a link to your GitHub repository or other public code-hosting
site.

This repository should contain the code for solving the homework.

When your solution has SQL or shell commands and not code
(e.g. python files) file format, include them directly in
the README file of your repository.


## Question 1. Understanding Docker images

Run docker with the `python:3.13` image. Use an entrypoint `bash` to interact with the container.

What's the version of `pip` in the image?

- **25.3**
- 24.3.1
- 24.2.1
- 23.3.1

SOLUTION:
```bash
docker run -it --entrypoint=bash python:3.13
pip --version
```

## Question 2. Understanding Docker networking and docker-compose

Given the following `docker-compose.yaml`, what is the `hostname` and `port` that pgadmin should use to connect to the postgres database?

```yaml
services:
  db:
    container_name: postgres
    image: postgres:17-alpine
    environment:
      POSTGRES_USER: 'postgres'
      POSTGRES_PASSWORD: 'postgres'
      POSTGRES_DB: 'ny_taxi'
    ports:
      - '5433:5432'
    volumes:
      - vol-pgdata:/var/lib/postgresql/data

  pgadmin:
    container_name: pgadmin
    image: dpage/pgadmin4:latest
    environment:
      PGADMIN_DEFAULT_EMAIL: "pgadmin@pgadmin.com"
      PGADMIN_DEFAULT_PASSWORD: "pgadmin"
    ports:
      - "8080:80"
    volumes:
      - vol-pgadmin_data:/var/lib/pgadmin

volumes:
  vol-pgdata:
    name: vol-pgdata
  vol-pgadmin_data:
    name: vol-pgadmin_data
```

- postgres:5433
- localhost:5432
- db:5433
- **postgres:5432**
- **db:5432**

If multiple answers are correct, select any 

## Prepare the Data

Download the green taxi trips data for November 2025:

```bash
wget https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-11.parquet
```

You will also need the dataset with zones:

```bash
wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/misc/taxi_zone_lookup.csv
```

In [1]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-11.parquet

--2026-01-22 15:07:06--  https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:20dc:1600:b:20a5:b140:21, 2600:9000:20dc:3200:b:20a5:b140:21, 2600:9000:20dc:a200:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:20dc:1600:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1164775 (1,1M) [binary/octet-stream]
Saving to: ‘green_tripdata_2025-11.parquet’

green_tripdata_2025 100%[===================>]   1,11M  --.-KB/s    in 0,03s   

2026-01-22 15:07:06 (40,1 MB/s) - ‘green_tripdata_2025-11.parquet’ saved [1164775/1164775]



In [2]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/misc/taxi_zone_lookup.csv

--2026-01-22 15:07:19--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/misc/taxi_zone_lookup.csv
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/5a2cc2f5-b4cd-4584-9c62-a6ea97ed0e6a?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-01-22T14%3A07%3A30Z&rscd=attachment%3B+filename%3Dtaxi_zone_lookup.csv&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-01-22T13%3A07%3A16Z&ske=2026-01-22T14%3A07%3A30Z&sks=b&skv=2018-11-09&sig=Qj9ID%2FI3IbDZPvpNlMfgP28wtsQe4pPhRoWCnwHpyk0%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc2OTA4NzUzOSwibmJmIjoxNzY5MDg3MjM5LCJwYXRoIjoicmVsZWFzZWFzc2V0c

## Question 3. Counting short trips

For the trips in November 2025 (lpep_pickup_datetime between '2025-11-01' and '2025-12-01', exclusive of the upper bound), how many trips had a `trip_distance` of less than or equal to 1 mile?

- 7,853
- **8,007**
- 8,254
- 8,421

In [7]:
import pandas as pd

In [8]:
# Read the parquet file using fastparquet engine
df_parquet = pd.read_parquet('./green_tripdata_2025-11.parquet', engine='fastparquet')

In [ ]:
df_parquet.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,2,2025-11-01 00:34:48,2025-11-01 00:41:39,N,1.0,74,42,1.0,0.74,7.2,...,0.5,1.94,0.0,NaN,1.0,11.64,1.0,1.0,0.00,0.0
1,2,2025-11-01 00:18:52,2025-11-01 00:24:27,N,1.0,74,42,2.0,0.95,7.2,...,0.5,0.00,0.0,NaN,1.0,9.70,2.0,1.0,0.00,0.0
2,2,2025-11-01 01:03:14,2025-11-01 01:15:24,N,1.0,83,160,1.0,2.19,13.5,...,0.5,5.00,0.0,NaN,1.0,21.00,1.0,1.0,0.00,0.0
3,2,2025-11-01 00:10:57,2025-11-01 00:24:53,N,1.0,166,127,1.0,5.44,24.7,...,0.5,0.50,0.0,NaN,1.0,27.70,1.0,1.0,0.00,0.0
4,1,2025-11-01 00:03:48,2025-11-01 00:19:38,N,1.0,166,262,1.0,3.20,18.4,...,1.5,1.00,0.0,NaN,1.0,24.65,1.0,1.0,2.75,0.0


In [10]:
df_parquet.dtypes

VendorID                          int32
lpep_pickup_datetime     datetime64[us]
lpep_dropoff_datetime    datetime64[us]
store_and_fwd_flag               object
RatecodeID                      float64
PULocationID                      int32
DOLocationID                      int32
passenger_count                 float64
trip_distance                   float64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
ehail_fee                       float64
improvement_surcharge           float64
total_amount                    float64
payment_type                    float64
trip_type                       float64
congestion_surcharge            float64
cbd_congestion_fee              float64
dtype: object

In [11]:
# Read the CSV file
df_csv = pd.read_csv('./taxi_zone_lookup.csv')

In [12]:
df_csv.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [13]:
df_csv.dtypes

LocationID       int64
Borough         object
Zone            object
service_zone    object
dtype: object

In [14]:
november_trips = df[
    (df_parquet['lpep_pickup_datetime'] >= '2025-11-01') & 
    (df_parquet['lpep_pickup_datetime'] < '2025-12-01') &
    (df_parquet['trip_distance'] <= 1.0)
]

print(f"Short trips count: {len(november_trips)}")

Short trips count: 8007


## Question 4. Longest trip for each day

Which was the pick up day with the longest trip distance? Only consider trips with `trip_distance` less than 100 miles (to exclude data errors).

Use the pick up time for your calculations.

- **2025-11-14**
- 2025-11-20
- 2025-11-23
- 2025-11-25

In [16]:
# Filter for November 2025 trips with distance < 100 miles
november_trips = df[
    (df_parquet['lpep_pickup_datetime'] >= '2025-11-01') & 
    (df_parquet['lpep_pickup_datetime'] < '2025-12-01') &
    (df_parquet['trip_distance'] < 100)
]

# Extract the pickup date (without time)
november_trips['pickup_date'] = november_trips['lpep_pickup_datetime'].dt.date

# Find the maximum trip distance for each day
daily_max = november_trips.groupby('pickup_date')['trip_distance'].max()

# Sort to see the top days
print(daily_max.sort_values(ascending=False).head(10))

# Find the day with the longest trip
longest_day = daily_max.idxmax()
longest_distance = daily_max.max()

print(f"\nDay with longest trip: {longest_day}")
print(f"Distance: {longest_distance} miles")

pickup_date
2025-11-14    88.03
2025-11-20    73.84
2025-11-23    45.26
2025-11-22    40.16
2025-11-15    39.81
2025-11-11    39.00
2025-11-19    38.68
2025-11-16    37.67
2025-11-10    36.46
2025-11-26    36.10
Name: trip_distance, dtype: float64

Day with longest trip: 2025-11-14
Distance: 88.03 miles


/var/folders/cf/yvx1ty9x60zg9rmfrmvm7pyc0000gn/T/ipykernel_8953/396806896.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  november_trips['pickup_date'] = november_trips['lpep_pickup_datetime'].dt.date


## Question 5. Biggest pickup zone

Which was the pickup zone with the largest `total_amount` (sum of all trips) on November 18th, 2025?

- **East Harlem North**
- East Harlem South
- Morningside Heights
- Forest Hills

In [20]:
# Filter for trips on November 18th, 2025
nov_18_trips = df_parquet[
    (df_parquet['lpep_pickup_datetime'] >= '2025-11-18') & 
    (df_parquet['lpep_pickup_datetime'] < '2025-11-19')
].copy()

# Join with zone lookup to get zone names
nov_18_with_zones = nov_18_trips.merge(
    df_csv, 
    left_on='PULocationID', 
    right_on='LocationID', 
    how='left'
)

# Group by zone and sum the total_amount
zone_totals = nov_18_with_zones.groupby('Zone')['total_amount'].sum().sort_values(ascending=False)

print("Top 10 pickup zones by total_amount on November 18th, 2025:")
print(zone_totals.head(10))

Top 10 pickup zones by total_amount on November 18th, 2025:
Zone
East Harlem North              9281.92
East Harlem South              6696.13
Central Park                   2378.79
Washington Heights South       2139.05
Morningside Heights            2100.59
Jamaica                        1998.11
Fort Greene                    1780.41
Downtown Brooklyn/MetroTech    1499.02
Forest Hills                   1423.75
Elmhurst                       1251.82
Name: total_amount, dtype: float64


## Question 6. Largest tip

For the passengers picked up in the zone named "East Harlem North" in November 2025, which was the drop off zone that had the largest tip?

Note: it's `tip` , not `trip`. We need the name of the zone, not the ID.

- JFK Airport
- **Yorkville West**
- East Harlem North
- LaGuardia Airport

In [21]:
# Filter for November 2025 trips
november_trips = df_parquet[
    (df_parquet['lpep_pickup_datetime'] >= '2025-11-01') & 
    (df_parquet['lpep_pickup_datetime'] < '2025-12-01')
].copy()

# Join with zones to get pickup zone names
november_with_pickup_zones = november_trips.merge(
    df_csv, 
    left_on='PULocationID', 
    right_on='LocationID', 
    how='left'
)

# Filter for pickups in "East Harlem North"
east_harlem_pickups = november_with_pickup_zones[
    november_with_pickup_zones['Zone'] == 'East Harlem North'
].copy()

# Join again to get drop-off zone names
east_harlem_with_dropoff = east_harlem_pickups.merge(
    df_csv, 
    left_on='DOLocationID', 
    right_on='LocationID', 
    how='left',
    suffixes=('_pickup', '_dropoff')
)

# Find the drop-off zone with the largest tip
largest_tip_idx = east_harlem_with_dropoff['tip_amount'].idxmax()
largest_tip_row = east_harlem_with_dropoff.loc[largest_tip_idx]

print(f"Drop-off zone with largest tip: {largest_tip_row['Zone_dropoff']}")
print(f"Tip amount: ${largest_tip_row['tip_amount']:.2f}")

# Show top 10 drop-off zones by tip amount
print("\n" + "="*50)
print("\nTop 10 drop-off zones by tip amount:")
top_tips = east_harlem_with_dropoff.nlargest(10, 'tip_amount')[['Zone_dropoff', 'tip_amount']]
print(top_tips.to_string(index=False))

Drop-off zone with largest tip: Yorkville West
Tip amount: $81.89


Top 10 drop-off zones by tip amount:
                 Zone_dropoff  tip_amount
               Yorkville West       81.89
            LaGuardia Airport       50.00
            East Harlem North       45.00
Long Island City/Queens Plaza       34.25
                          NaN       28.90
            East Harlem North       26.00
                  JFK Airport       23.53
               Newark Airport       20.00
          Morningside Heights       20.00
            East Harlem South       20.00


## Terraform

In this section homework we'll prepare the environment by creating resources in GCP with Terraform.

In your VM on GCP/Laptop/GitHub Codespace install Terraform.
Copy the files from the course repo
[here](../../../01-docker-terraform/terraform/terraform) to your VM/Laptop/GitHub Codespace.

Modify the files as necessary to create a GCP Bucket and Big Query Dataset.


## Question 7. Terraform Workflow

Which of the following sequences, respectively, describes the workflow for:
1. Downloading the provider plugins and setting up backend,
2. Generating proposed changes and auto-executing the plan
3. Remove all resources managed by terraform`

Answers:
- terraform import, terraform apply -y, terraform destroy
- teraform init, terraform plan -auto-apply, terraform rm
- terraform init, terraform run -auto-approve, terraform destroy
- **terraform init, terraform apply -auto-approve, terraform destroy**
- terraform import, terraform apply -y, terraform rm